# 第7回：移動表現の分類器を作って評価する

このノートブックは[第7回の授業ページ](https://nlp.lang.osaka-u.ac.jp/ja/dh/b/2026-11-24/)の3〜7節に対応します。上から順にセルを実行し、「**編集**」と記した六つのセルを書き換えて進めてください。

**AIツールの利用方針：**3〜6節では、Pythonのエラー表示の説明にAIを使ってかまいません。実行前の予想や、3節・4節・7節での本文の読み取りと判断は、AIに書かせず自分で行ってください。

### 準備（実行のみ）
この折りたたみセルを一度実行してください。第7回の教材ZIPを取得し、Pythonのリスト（`train_rows`・`dev_rows`・`test_rows`）と補助関数を読み込みます。中身を読んだり書き換えたりする必要はありません。

In [ ]:
#@title 本文・ラベル・埋め込み・補助関数を読み込む（実行のみ）
import json
from collections import Counter
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

LABELS = ["CT+", "CT-", "Uu", "NM"]
url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)

with ZipFile(archive) as bundle:
    all_passages = [json.loads(line) for line in bundle.open("class7/passages.jsonl")]
    raw_labels = [json.loads(line) for line in bundle.open("class7/llm-labels.jsonl")]
    llm_labels = {row["id"]: row["label"] for row in raw_labels if row["label"] in LABELS}
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vec_map = dict(zip(saved["ids"].tolist(), saved["vectors"].astype("float32")))
    saved_predictions = {}
    for method in ("majority", "rules", "gliner", "gliner-ft", "zeroshot-2b", "lora-2b"):
        lines = [json.loads(line) for line in bundle.open(f"class7/predictions/{method}.jsonl")]
        saved_predictions[method] = {row["id"]: row["pred"] for row in lines[1:]}

for row in all_passages:
    row["reference"] = llm_labels.get(row["id"])

train_rows = [row for row in all_passages if row["split"] == "train" and row["reference"]]
dev_rows = [row for row in all_passages if row["split"] == "dev" and row["reference"]]
test_rows = [row for row in all_passages if row["split"] == "test" and row["reference"]]
by_id = {row["id"]: row for row in all_passages}

train_vectors = np.stack([vec_map[row["id"]] for row in train_rows])
dev_vectors = np.stack([vec_map[row["id"]] for row in dev_rows])
test_vectors = np.stack([vec_map[row["id"]] for row in test_rows])
train_labels = [row["reference"] for row in train_rows]
dev_labels = [row["reference"] for row in dev_rows]
test_labels = [row["reference"] for row in test_rows]

lemma_votes = {}
for row in train_rows:
    lemma_votes.setdefault(row["lemma"], Counter())[row["reference"]] += 1
lemma_majority = {lemma: counts.most_common(1)[0][0] for lemma, counts in lemma_votes.items()}

_order = np.random.default_rng(20261124).permutation(len(train_rows))
shuffled_vectors = train_vectors[_order]
shuffled_labels = [train_labels[i] for i in _order]

AUDIT_IDS = ("train-0407", "train-0569", "train-0360")
AUDIT_NOTES = {
    "train-0407": ("Uu", "「おらぬかと」は見回した理由です。金が落ちたかは分かりません。"),
    "train-0569": ("CT-", "「上って来ない」は今回のことです。次の文の「上って来た」は以前の出来事です。"),
    "train-0360": ("保留", "抽出された「降」だけでは判定を保留しました。「降懸る」全体を対象にすると、虫が降りかかるのでCT+と読めます。"),
}

def show_passage(row):
    print(f'{row["author"]}「{row["title"]}」文{row["sentence"]}（{row["id"]}）')
    print(row["previous"])
    print(row["left"] + "【" + row["target"] + "】" + row["right"])
    print(row["next"])

def show_audit_comparison(passage_id):
    reader_label, note = AUDIT_NOTES[passage_id]
    print(f"{passage_id}：Qwenは{by_id[passage_id]['reference']}・AIと相談した読者1名は{reader_label}")
    print(" ", note)

def saved_list(method, rows):
    return [saved_predictions[method][row["id"]] for row in rows]

def count_agreements(gold, predicted):
    matches = 0
    for g, p in zip(gold, predicted):
        if g == p:
            matches += 1
    return matches

def show_scores(name, gold, predicted):
    matches = count_agreements(gold, predicted)
    f1s = []
    recalls = {}
    for label in LABELS:
        tp = sum(g == label and p == label for g, p in zip(gold, predicted))
        n_pred = sum(p == label for p in predicted)
        n_gold = sum(g == label for g in gold)
        prec = tp / n_pred if n_pred else 0.0
        rec = tp / n_gold if n_gold else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        recalls[label] = (tp, n_gold)
        if n_gold:
            f1s.append(f1)
    macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0
    nm_tp, nm_n = recalls["NM"]
    uu_tp, uu_n = recalls["Uu"]
    print(f"{name}：{matches}/{len(gold)}｜マクロF1 {macro_f1:.3f}｜NM {nm_tp}/{nm_n}｜Uu {uu_tp}/{uu_n}")

def show_confusion_matrix(gold, predicted):
    counts = Counter(zip(gold, predicted))
    header = "基準 ＼ 予測 | " + " | ".join(f"{label:>4}" for label in LABELS)
    print(header)
    print("-" * len(header))
    for g in LABELS:
        cells = " | ".join(f"{counts[(g, p)]:>4}" for p in LABELS)
        print(f"{g:>10} | {cells}")

def show_top_cues(vectorizer, model, label, k=8):
    classes = list(model.classes_)
    feats = vectorizer.get_feature_names_out()
    weights = model.coef_[classes.index(label)]
    order = np.argsort(weights)[::-1][:k]
    items = [f"{feats[i]}（+{weights[i]:.2f}）" for i in order if weights[i] > 0]
    print(f"{label}の上位手掛かり：", "、".join(items))

def test_sentence(model, vectorizer, sentence, window=10):
    start = sentence.find("【")
    end = sentence.find("】", start + 1) if start >= 0 else -1
    if start >= 0 and end > start + 1:
        clipped = sentence[:start][-window:] + sentence[start:end + 1] + sentence[end + 1:][:window] if window > 0 else sentence[start:end + 1]
    else:
        clipped = sentence
    x = vectorizer.transform([clipped])
    classes = list(model.classes_)
    probs = model.predict_proba(x)[0]
    best = int(np.argmax(probs))
    pred = classes[best]
    prob_text = "、".join(f"{label} {probs[classes.index(label)] * 100:.0f}%" for label in LABELS if label in classes)
    feats = vectorizer.get_feature_names_out()
    contribs = [(float(x[0, j] * model.coef_[best, j]), feats[j]) for j in x.nonzero()[1]]
    top_pos = [f"{gram}（+{w:.2f}）" for w, gram in sorted(contribs, reverse=True)[:4] if w > 0.005]
    print("計算範囲：", clipped)
    print("モデルの予測：", pred, "｜", prob_text)
    print("上位の手掛かり：", "、".join(top_pos))

def build_hybrid_features(rows, vectors, encoder=None):
    cue_dicts = []
    for row in rows:
        right = row["right"]
        cue_dicts.append({
            "lemma": row["lemma"],
            "negative": right.startswith(("ない", "なかっ")),
            "conditional": right.startswith(("なら", "ば", "たら")),
        })
    if encoder is None:
        encoder = DictVectorizer(sparse=False)
        cue_matrix = encoder.fit_transform(cue_dicts)
    else:
        cue_matrix = encoder.transform(cue_dicts)
    return np.column_stack((vectors, cue_matrix)), encoder

print(f"読み込み完了：学習用{len(train_rows)}件・検証用{len(dev_rows)}件・評価用{len(test_rows)}件")

### 3節　作品ごとの分割を確かめ、学習用の3箇所を読む
分類器が作品固有の名前や語彙を手掛かりにしないよう、一つの作品は一つの分割だけに入れます。次のセルで作品数と重なりが0件であることを確認してください。

In [ ]:
train_works = set()
for row in train_rows:
    train_works.add(row["file"])

test_works = set()
for row in test_rows:
    test_works.add(row["file"])

shared_works = train_works & test_works
print("学習用の作品数：", len(train_works))
print("評価用の作品数：", len(test_works))
print("両方に含まれる作品数：", len(shared_works))

続く学習用の3箇所を読み、自分のラベル（`CT+`・`CT-`・`Uu`・`NM`、または対象語の範囲を要確認）と判断に使った語句をメモしてから、比較セルを実行してください。

In [ ]:
for passage_id in AUDIT_IDS:
    show_passage(by_id[passage_id])
    print()

In [ ]:
for passage_id in AUDIT_IDS:
    show_audit_comparison(passage_id)

### 4節　語彙素から予測し、左文脈の規則を試す
`dev-0054`を読んでください。何が落ちるのでしょうか。規則を実行する前に、自分の予想（`CT+`・`CT-`・`Uu`・`NM`）をメモしてください。

In [ ]:
show_passage(by_id["dev-0054"])

**編集1**　語彙素ごとの多数派を出発点とし、語彙素が「落ちる」で左文脈が`cue`で終わるときだけ`NM`に変える規則です。まず`cue = "眠りに"`のまま次の二つのセルを実行してください。次に`dev-0054`で何が落ちるかを示す「が」で終わる語句へ書き換え、検証用59箇所全体で一致件数がどう変わるか確認してください。

In [ ]:
cue = "眠りに"

In [ ]:
rule_predictions = []
changed_ids = []

for row in dev_rows:
    prediction = lemma_majority.get(row["lemma"], "CT+")
    left = row["left"]
    if row["lemma"] == "落ちる" and left.endswith(cue):
        prediction = "NM"
        changed_ids.append(row["id"])
    rule_predictions.append(prediction)

matched = count_agreements(dev_labels, rule_predictions)
print("手掛かり：", cue)
print("予測が変わったID：", changed_ids)
print("検証用での一致：", matched, "/", len(dev_rows))

続けて`dev-0046`（「本国の土佐へ【落ち】て行った」）を前後の文と読んでください。上の規則の予測と自分のラベルをメモしてから、Qwenのラベルを開いてください。

In [ ]:
show_passage(by_id["dev-0046"])

In [ ]:
print("Qwen：", by_id["dev-0046"]["reference"])

### 5節A　文字nグラム分類器を学習させ、前後文字数を変える
`【対象語】`の前後の文字を特徴にしてロジスティック回帰を学習します（`min_df=2`は学習用で2箇所以上に現れる文字のまとまりを残します）。**編集2**　まず`window = 10`で実行し、次に`window = 0`（動詞のみ）と`window = 200`（文全体）を予想してから試してください。5節B・6節に進む前に`window = 10`へ戻して再実行してください。

In [ ]:
window = 10

In [ ]:
def make_window(row, width):
    if width <= 0:
        return "【" + row["target"] + "】"
    left_part = row["left"][-width:]
    right_part = row["right"][:width]
    return left_part + "【" + row["target"] + "】" + right_part

train_texts = []
for row in train_rows:
    train_texts.append(make_window(row, window))

dev_texts = []
for row in dev_rows:
    dev_texts.append(make_window(row, window))

vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=2)
x_train_char = vectorizer.fit_transform(train_texts)
x_dev_char = vectorizer.transform(dev_texts)

char_model = LogisticRegression(max_iter=2000, C=1.0)
char_model.fit(x_train_char, train_labels)

char_dev_preds = char_model.predict(x_dev_char)
print("前後文字数：", window, "｜検証用での一致：", count_agreements(dev_labels, char_dev_preds), "/", len(dev_rows))
show_scores("文字nグラム（検証用）", dev_labels, char_dev_preds)
show_confusion_matrix(dev_labels, char_dev_preds)
show_top_cues(vectorizer, char_model, "CT+")
show_top_cues(vectorizer, char_model, "NM")

### 5節B　比べる例文と判断が分かれた学習例で弱点を探す
まず授業ページの階段のペア（「階段を【上っ】た。」と「階段を【上ろ】うとした。」）の予測を予想してください。続けて「畜犬談」（`train-0281`：Qwenは`NM`、読者1名は`CT+`）を読み、「垂れる」が動きと姿勢のどちらを表すか比べてください。**編集3**　次のセルの`my_sentence`を書き換え、自分の予想を試してください。

In [ ]:
show_passage(by_id["train-0281"])

In [ ]:
test_sentence(char_model, vectorizer, "階段を【上っ】た。", window=window)
print()
test_sentence(char_model, vectorizer, "階段を【上ろ】うとした。", window=window)
print()
test_sentence(char_model, vectorizer, "犬は首を【垂れ】た。", window=window)
print()
test_sentence(char_model, vectorizer, "枝から縄が【垂れ】ている。", window=window)

In [ ]:
my_sentence = "犬は首を【垂れ】ている。"
test_sentence(char_model, vectorizer, my_sentence, window=window)

### 5節C　埋め込みで学習し、学習件数と重みを比べる
保存済みの埋め込み（1,024次元）でロジスティック回帰を学習します。**編集4**　`n_train`を`32`・`128`・`420`、`C_value`を`0.1`・`1.0`、`use_balanced`を`False`（標準）・`True`（少数ラベルを重く）で比べ、検証用のマクロF1と混同行列の変化を確認してください。6節に進む前に`n_train = 420`・`C_value = 1.0`・`use_balanced = False`へ戻してください。

In [ ]:
n_train = 420
C_value = 1.0
use_balanced = False

In [ ]:
sub_vectors = shuffled_vectors[:n_train]
sub_labels = shuffled_labels[:n_train]
weights = "balanced" if use_balanced else None

embed_model = LogisticRegression(max_iter=2000, C=C_value, class_weight=weights)
embed_model.fit(sub_vectors, sub_labels)
embed_dev_preds = embed_model.predict(dev_vectors)

print(f"埋め込みのみ（n={n_train}・C={C_value}・少数ラベルを重く={use_balanced}）：", count_agreements(dev_labels, embed_dev_preds), "/", len(dev_rows))
show_scores("埋め込み（検証用）", dev_labels, embed_dev_preds)
show_confusion_matrix(dev_labels, embed_dev_preds)

x_train_hybrid, cue_encoder = build_hybrid_features(train_rows, train_vectors)
x_dev_hybrid, cue_encoder_dev = build_hybrid_features(dev_rows, dev_vectors, cue_encoder)
hybrid_model = LogisticRegression(max_iter=2000, C=1.0)
hybrid_model.fit(x_train_hybrid, train_labels)
hybrid_dev_preds = hybrid_model.predict(x_dev_hybrid)
show_scores("埋め込み＋語尾の手掛かり（検証用）", dev_labels, hybrid_dev_preds)

**編集5**　授業ページのNMしきい値スライダーを検証用59箇所で再現します。`nm_cutoff`を`0.50`から`0.30`に下げると、読む候補の件数と拾えるNMの件数がどう変わるか予想してから書き換えて実行してください。

In [ ]:
nm_cutoff = 0.50

In [ ]:
nm_index = list(embed_model.classes_).index("NM")
dev_probs = embed_model.predict_proba(dev_vectors)

on_list = 0
nm_found = 0
nm_total = 0

for row, probs in zip(dev_rows, dev_probs):
    is_nm = row["reference"] == "NM"
    if is_nm:
        nm_total += 1
    if probs[nm_index] >= nm_cutoff:
        on_list += 1
        if is_nm:
            nm_found += 1

print("しきい値：", nm_cutoff)
print("読む候補の件数：", on_list, "/", len(dev_rows))
print("NMを拾えた件数：", nm_found, "/", nm_total)

### 6節　評価用の作品で比べる
選んだモデルの設定とNMの下限、それらを選ぶ根拠となる検証結果を一つメモしてください。次に評価用150箇所で各方法の一致数・マクロF1・`NM`と`Uu`の再現件数を比べます。

In [ ]:
test_texts = []
for row in test_rows:
    test_texts.append(make_window(row, window))

char_test_preds = char_model.predict(vectorizer.transform(test_texts))
embed_test_preds = embed_model.predict(test_vectors)
x_test_hybrid, cue_encoder_test = build_hybrid_features(test_rows, test_vectors, cue_encoder)
hybrid_test_preds = hybrid_model.predict(x_test_hybrid)

rules_test_preds = saved_list("rules", test_rows)
gliner_ft_test_preds = saved_list("gliner-ft", test_rows)
lora_2b_test_preds = saved_list("lora-2b", test_rows)

show_scores("第4回の語尾の規則", test_labels, rules_test_preds)
show_scores("文字nグラム（現在の窓）", test_labels, char_test_preds)
show_scores("埋め込み（現在の設定）", test_labels, embed_test_preds)
show_scores("埋め込み＋語尾の手掛かり", test_labels, hybrid_test_preds)
show_scores("GLiNER2（学習後・GPU）", test_labels, gliner_ft_test_preds)
show_scores("Qwen3.5-2B（LoRA・GPU）", test_labels, lora_2b_test_preds)

### 7節　評価データで不一致を読む
授業ページに合わせて、まず2BのLoRAで`Uu`の不一致を確認します（`method_to_check = "lora-2b"`・`reference_filter = "Uu"`）。続けて`method_to_check = "rules"`（語尾の規則）や`reference_filter = "ALL"`（全ラベル）にも切り替えて比べてください。**編集6**　条件と`case_number`を選び、本文を読んで自分のラベルと根拠の語句をメモしてからラベル表示セルを実行してください。

In [ ]:
method_to_check = "lora-2b"
reference_filter = "Uu"
case_number = 0

In [ ]:
method_preds = {
    "rules": rules_test_preds,
    "char": list(char_test_preds),
    "embed": list(embed_test_preds),
    "gliner-ft": gliner_ft_test_preds,
    "lora-2b": lora_2b_test_preds,
}
chosen_preds = method_preds[method_to_check]

disagreements = []
for idx, row in enumerate(test_rows):
    pred = chosen_preds[idx]
    matches_filter = (reference_filter == "ALL") or (row["reference"] == reference_filter)
    if pred != row["reference"] and matches_filter:
        disagreements.append((row, char_test_preds[idx], embed_test_preds[idx], rules_test_preds[idx], lora_2b_test_preds[idx]))

print("方法：", method_to_check, "｜絞り込み：", reference_filter, "｜不一致件数：", len(disagreements))
show_confusion_matrix(test_labels, chosen_preds)

In [ ]:
case_row, case_char_pred, case_embed_pred, case_rules_pred, case_lora_pred = disagreements[case_number]
show_passage(case_row)

自分の読みと判断の根拠をメモしてからラベルを開いてください。前後の文から支持できるのはどの読みですか。

In [ ]:
print("Qwen（暫定）：  ", case_row["reference"])
print("第4回の規則：   ", case_rules_pred)
print("文字分類器：    ", case_char_pred)
print("埋め込み分類器：", case_embed_pred)
print("Qwen3.5-2B LoRA：", case_lora_pred)